# Multi-seed validation of the operational foundation-model panel (Axis 1)

**What this notebook does.** Runs the operational foundation-model benchmark from the paper across multiple random splits, then reports seed-mean accuracy with bootstrap 95 % confidence intervals and paired Wilcoxon p-values for cell-by-cell comparisons.

**Why.** The single-seed protocol used in `models_use_example.ipynb` cannot test small accuracy gaps for statistical significance — for example, the +0.97 pp TabPFN → TabPFN 2.5 gain on CIC-IDS2017 V4. The protocol implemented here resamples the train/test split itself across N seeds and yields the seed-level distributions on which non-parametric tests are valid.

**Recommended budget.** Five seeds is the standard. Three seeds is a faster floor with wider CIs. Two seeds is enough to spot bugs but does not give a meaningful CI.

**Compute notes.**
- TabPFN, TabPFN 2.5, TabICL, Random Forest: cheap. Five seeds on one (dataset, variant) cell takes minutes on a single H200.
- TabSTAR: each seed includes a LoRA fine-tune (50 epochs, patience 5). Five seeds on one cell is ~5× the single-seed cost. Plan accordingly; the runner can be invoked with `seeds=(0, 1)` for a TabSTAR-only sanity check before scaling up.


In [ ]:
from src.data import CICIDS2017, N_BaIoT, CIC_UNSW
from src.pipelines import MultiSeedRunner, compare_runners, run_panel

import logging
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

## 1. Choose a (dataset, variant) cell

The Axis-1 panel iterates over (dataset × variant) cells. Pick one to start; below uses **CIC-IDS2017 V1** (no PCA, no class mapping). Replace with `pca=True` / `classes_mapping=True` for the other variants, or swap the dataset constructor for `N_BaIoT(...)` / `CIC_UNSW(pca=...)`.

In [ ]:
dataset = CICIDS2017(pca=False, classes_mapping=False)  # V1 = (no PCA, no class mapping)
dataset.load()
print(f'Loaded {len(dataset.data):,} rows, {dataset.data.shape[1]-1} features, '
      f'{dataset.data["Label"].nunique()} classes.')

## 2. TabPFN vs TabPFN 2.5 (paired Wilcoxon)

We instantiate two `MultiSeedRunner` objects, one per release of TabPFN, both with the same seed list. Because the seeds match across runners, the resulting per-seed accuracies are paired and we can run a Wilcoxon signed-rank test on the differences.

In [ ]:
SEEDS = [0, 1, 2, 3, 4]   # paper's Axis-2 protocol
TEST_SIZE = 0.3            # paper's Axis-1 protocol
FM_BUDGET = 2000           # 2,000 rows / class for TabPFN family + TabICL

runner_tabpfn = MultiSeedRunner(
    dataset=dataset,
    model_name='tabpfn',
    model_kwargs={'version': '2'},
    seeds=SEEDS,
    test_size=TEST_SIZE,
    memory_bound_per_class=FM_BUDGET,
)
runner_tabpfn.run_all()

In [ ]:
runner_tabpfn25 = MultiSeedRunner(
    dataset=dataset,
    model_name='tabpfn_2.5',   # default version inside seedutils
    seeds=SEEDS,
    test_size=TEST_SIZE,
    memory_bound_per_class=FM_BUDGET,
)
runner_tabpfn25.run_all()

In [ ]:
summary_a = runner_tabpfn.summary()
summary_b = runner_tabpfn25.summary()

print(f'TabPFN     : acc = {summary_a["accuracy_mean"]:.4f}  '
      f'95% CI [{summary_a["accuracy_lo95"]:.4f}, {summary_a["accuracy_hi95"]:.4f}]')
print(f'TabPFN 2.5 : acc = {summary_b["accuracy_mean"]:.4f}  '
      f'95% CI [{summary_b["accuracy_lo95"]:.4f}, {summary_b["accuracy_hi95"]:.4f}]')

In [ ]:
cmp = compare_runners(runner_tabpfn25, runner_tabpfn, metric='accuracy')
print('Paired Wilcoxon (TabPFN 2.5 vs TabPFN):')
print(f'  mean diff      = {cmp["mean_diff"]:+.4f}')
print(f'  Wilcoxon W     = {cmp["wilcoxon_stat"]}')
print(f'  p-value (2-sided) = {cmp["wilcoxon_p"]:.4f}')
print(f'  n_seeds        = {cmp["n_seeds"]}')

**Reading the result.** If the paired Wilcoxon p-value is `< 0.05`, the per-seed accuracy distributions of TabPFN and TabPFN 2.5 are distinguishable on this configuration; if not, the +0.97 pp gain reported in the single-seed table is not statistically reliable.

Five seeds is the smallest reasonable Wilcoxon n. If the p-value sits in the marginal region (0.05 ≤ p ≤ 0.15), increase to ten seeds before drawing strong conclusions.

## 3. Persist results

Each runner can save a per-seed CSV (audit trail) and an aggregate JSON (CIs, per-class mean F1) to disk. Use a directory naming convention that captures the (dataset, variant) cell so runs accumulate cleanly across cells.

In [ ]:
out_dir = 'multiseed_runs/cic_ids2017_V1'
runner_tabpfn.save(out_dir)
runner_tabpfn25.save(out_dir)

## 4. The full operational foundation-model panel

When you are ready to reproduce one row of Table 8 of the paper, use `run_panel`. The model_specs list mirrors Table 1 of the paper:

- Random Forest: classical anchor (no FM budget)
- TabPFN: legacy release (2,000 rows / class)
- TabPFN 2.5: new entrant (2,000 rows / class)
- TabICL: scaled in-context learner (2,000 rows / class)
- TabSTAR: text-encoder model (4,000 rows / class)


In [ ]:
model_specs = [
    {'name': 'random_forest'},
    {'name': 'tabpfn',     'model_kwargs': {'version': '2'},   'memory_bound_per_class': 2000},
    {'name': 'tabpfn_2.5',                                            'memory_bound_per_class': 2000},
    {'name': 'tabicl',                                                'memory_bound_per_class': 2000},
    {'name': 'tabstar',                                               'memory_bound_per_class': 4000},
]

runners = run_panel(
    dataset=dataset,
    model_specs=model_specs,
    seeds=SEEDS,
    out_dir='multiseed_runs/cic_ids2017_V1',
    test_size=TEST_SIZE,
)

In [ ]:
import pandas as pd
rows = []
for name, runner in runners.items():
    s = runner.summary()
    rows.append({
        'model':       name,
        'acc_mean':    s['accuracy_mean'],
        'acc_lo95':    s['accuracy_lo95'],
        'acc_hi95':    s['accuracy_hi95'],
        'macroF1_mean':s['macro_f1_mean'],
        'n_seeds':     s['n_seeds'],
    })
panel_df = pd.DataFrame(rows).sort_values('acc_mean', ascending=False).reset_index(drop=True)
panel_df.round(4)

In [ ]:
# Pairwise comparisons against Random Forest, the natural anchor.
rf_runner = runners['random_forest']
cmp_rows = []
for name, runner in runners.items():
    if name == 'random_forest':
        continue
    cmp = compare_runners(runner, rf_runner, metric='accuracy')
    cmp_rows.append({
        'model':         name,
        'mean_diff_vs_RF': cmp['mean_diff'],
        'wilcoxon_p':    cmp['wilcoxon_p'],
        'n_seeds':       cmp['n_seeds'],
    })
pd.DataFrame(cmp_rows).sort_values('mean_diff_vs_RF', ascending=False).round(4)

## 5. Reproducing every cell of Table 8

Wrap the cells above in two outer loops:
```python
for dataset_name in ['cic_ids2017', 'n_baiot', 'cic_unsw']:
    for variant in ['V1', 'V2', 'V3', 'V4']:
        if dataset_name == 'cic_unsw' and variant in ('V3', 'V4'):
            continue   # CIC-UNSW only reports V1, V2 in the paper
        ds = build_dataset(dataset_name, variant)
        ds.load()
        run_panel(
            dataset=ds,
            model_specs=model_specs,
            seeds=[0, 1, 2, 3, 4],
            out_dir=f'multiseed_runs/{dataset_name}_{variant}',
            test_size=0.3,
        )
```

After all cells finish, every `multiseed_runs/<dataset>_<variant>/<model>_summary.json` aggregates the per-seed metrics for that cell. Stack them into a single dataframe to produce the seed-mean version of Table 8 with bootstrap CIs and paired Wilcoxon p-values for every model-vs-RF and TabPFN-vs-TabPFN-2.5 comparison.